# PNG Gating for Text Transformers — BERT-base (HateXplain, Frozen Backbone)
## Hate Speech Detection · HateXplain Dataset

Adapts **Patch-Norm Gating (PNG)** from Vision Transformers to **BERT-base-uncased**
for 3-class hate speech classification on the HateXplain dataset.

**Key setup:**
- Model: `bert-base-uncased` (110 M params, **12 layers**, 12 heads)
- Backbone **frozen** — only `classifier`, `bert.pooler`, and `gate_params` train
- Higher LR (5e-4) for the small trainable head
- More epochs (10) — frozen backbone needs more passes to converge

| | |
|---|---|
| **Model** | `bert-base-uncased` (110 M params, 12 layers, 12 heads) |
| **Dataset** | HateXplain · `Hate-speech-CNERG/hatexplain` |
| **Task** | 3-class: `hatespeech` (0) / `normal` (1) / `offensive` (2) |
| **Backbone** | Frozen — only classifier, pooler, gate params train |
| **Novel** | Token-norm-aware gating (PNG) adapted from ViT patch gating |

## 0. Setup

In [ ]:
import os, types, math, random
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
)
from datasets import load_dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, classification_report,
)
import pandas as pd

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── TEST MODE ─────────────────────────────────────────────────────────────────
# Set TEST = True to do a quick sanity run on 1% of data (1 epoch per variant).
# Set TEST = False for the real training run.
TEST = True

MAX_LEN  = 128
BATCH    = 32 if not TEST else 8
EPOCHS   = 10 if not TEST else 1     # frozen backbone needs more passes
LR_CLS   = 5e-4
NUM_CLS  = 3            # hatespeech / normal / offensive
CKPT_DIR = Path('checkpoints_hatexplain_frozen' + ('_test' if TEST else ''))
CKPT_DIR.mkdir(exist_ok=True)

LABEL_NAMES = ['hatespeech', 'normal', 'offensive']

def _safe_name(n):
    return n.replace(' ', '_').replace('-', '').replace('(', '').replace(')', '')

print(f'torch {torch.__version__} | device {DEVICE}')
print(f'TEST MODE: {TEST}')
print(f'Checkpoints -> {CKPT_DIR.resolve()}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint restore — copy from previous Kaggle version output if available
#
# On Kaggle: attach the output of the previous notebook version as an input
# dataset (Edit -> Add data -> Your work -> this notebook -> version N).
# The files will appear under /kaggle/input/<notebook-slug>/checkpoints_hatexplain_frozen/
# This cell finds them automatically and copies to the current working dir.
# ─────────────────────────────────────────────────────────────────────────────
import shutil, glob as _glob

CKPT_DIR = Path('checkpoints_hatexplain_frozen')
CKPT_DIR.mkdir(exist_ok=True)

# Search all mounted input datasets for a checkpoints_hatexplain_frozen folder
_src_dirs = _glob.glob('/kaggle/input/**/checkpoints_hatexplain_frozen', recursive=True)

if _src_dirs:
    _src = Path(sorted(_src_dirs)[-1])  # pick latest if multiple
    print(f'Found checkpoint source: {_src}')
    _copied = 0
    for _f in _src.iterdir():
        _dst = CKPT_DIR / _f.name
        if not _dst.exists():          # don't overwrite locally produced files
            shutil.copy2(_f, _dst)
            print(f'  Copied: {_f.name}')
            _copied += 1
        else:
            print(f'  Skipped (exists): {_f.name}')
    print(f'Done — {_copied} file(s) copied.')
else:
    print('No previous checkpoint directory found under /kaggle/input/.')
    print('To restore checkpoints: Edit -> Add data -> Your work -> this notebook -> version N')

print(f'\nCheckpoints in working dir:')
for _f in sorted(CKPT_DIR.iterdir()):
    print(f'  {_f.name}  ({_f.stat().st_size/1e6:.1f} MB)')

## 1. Dataset & Tokenizer

**HateXplain** provides token-level rationale annotations from 3 crowd-workers per post,
in addition to utterance-level labels (hatespeech / normal / offensive).  
We derive:
- **Majority label** via `Counter` over 3 annotator labels
- **Word-level rationale mask** by averaging the 3 annotators' binary rationale lists and thresholding at 0.5

In [ ]:
# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ── Load HateXplain from official GitHub JSON ─────────────────────────────────
# Source: https://github.com/hate-alert/HateXplain
# Using GitHub directly — the HF version uses a legacy script that no longer loads.
import urllib.request as _req
import json as _json

_URL       = 'https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json'
_SPLIT_URL = 'https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/post_id_divisions.json'

print('Downloading HateXplain dataset from GitHub...')
with _req.urlopen(_URL) as r:
    _data = _json.loads(r.read().decode())
with _req.urlopen(_SPLIT_URL) as r:
    _splits = _json.loads(r.read().decode())

print(f'Total posts: {len(_data)}')
print(f'Train: {len(_splits["train"])} | Val: {len(_splits["val"])} | Test: {len(_splits["test"])}')

# Show a sample entry
_sid = _splits['train'][0]
print('\nSample post_id:', _sid)
print('Sample data:', _data[_sid])

# ── Label mapping ──────────────────────────────────────────────────────────────
# annotators[i]['label'] is a string: 'hatespeech' | 'normal' | 'offensive'
_LABEL2IDX = {'hatespeech': 0, 'normal': 1, 'offensive': 2}

def majority_label(annotators):
    labels = [a['label'] for a in annotators]
    return _LABEL2IDX[Counter(labels).most_common(1)[0][0]]

def avg_rationale(ex):
    """Average rationale masks from the post-level 'rationales' list; threshold at 0.5."""
    n_tokens = len(ex['post_tokens'])
    masks = ex.get('rationales', [])
    if not masks:
        return [0] * n_tokens
    padded = []
    for rat in masks:
        rat = list(rat)
        if len(rat) < n_tokens:
            rat = rat + [0] * (n_tokens - len(rat))
        else:
            rat = rat[:n_tokens]
        padded.append(rat)
    avg = np.mean(padded, axis=0)
    return (avg >= 0.5).astype(int).tolist()

# ── Dataset class ──────────────────────────────────────────────────────────────
class HateXplainDataset(Dataset):
    def __init__(self, post_ids, data_dict, subset_frac=None):
        ids = list(post_ids)
        if subset_frac is not None and subset_frac < 1.0:
            k = max(1, int(len(ids) * subset_frac))
            rng = random.Random(SEED)
            ids = rng.sample(ids, k)
        self.samples = []
        for pid in ids:
            ex        = data_dict[pid]
            tokens    = ex['post_tokens']
            text      = ' '.join(tokens)
            label     = majority_label(ex['annotators'])
            rationale = avg_rationale(ex)
            enc = tokenizer(
                text,
                truncation=True,
                max_length=MAX_LEN,
                padding='max_length',
                return_tensors='pt',
            )
            self.samples.append({
                'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label':          torch.tensor(label, dtype=torch.long),
                'post_tokens':    tokens,
                'rationale':      rationale,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'label':          torch.stack([b['label']          for b in batch]),
        'post_tokens':    [b['post_tokens'] for b in batch],
        'rationale':      [b['rationale']   for b in batch],
    }


# ── Build splits — use 1% in TEST mode ────────────────────────────────────────
frac = 0.01 if TEST else None
print(f'\nBuilding datasets (TEST={TEST}, frac={frac})...')
train_ds = HateXplainDataset(_splits['train'], _data, subset_frac=frac)
val_ds   = HateXplainDataset(_splits['val'],   _data, subset_frac=frac)
test_ds  = HateXplainDataset(_splits['test'],  _data, subset_frac=frac)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, collate_fn=collate_fn)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, collate_fn=collate_fn)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, collate_fn=collate_fn)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
if TEST:
    print('[TEST MODE] Using 1% of data — set TEST=False for full training run.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset Characteristics Analysis  (runs on full dataset regardless of TEST mode)
# ─────────────────────────────────────────────────────────────────────────────
_full_train_ids = _splits['train']
_full_val_ids   = _splits['val']
_full_test_ids  = _splits['test']

def _get_labels(ids):
    return [majority_label(_data[i]['annotators']) for i in ids]

print('=' * 60)
print('DATASET: HateXplain')
print('Source : Twitter + Gab  |  3 annotators per post')
print('Task   : 3-class — hatespeech / normal / offensive')
print('Labels : Majority vote from 3 crowd-worker annotations')
print('Paper  : Mathew et al., AAAI 2021  (arXiv:2012.10289)')
print('License: CC-BY 4.0')
print('=' * 60)
print()

# ── 1. Class distribution ─────────────────────────────────────────────────────
print('── 1. Class Distribution ──')
for split_name, ids in [('TRAIN', _full_train_ids), ('VALIDATION', _full_val_ids), ('TEST', _full_test_ids)]:
    n      = len(ids)
    labels = _get_labels(ids)
    print(f'  {split_name} ({n} samples):')
    for cls_idx, cls_name in enumerate(LABEL_NAMES):
        cnt = labels.count(cls_idx)
        bar = '#' * int(40 * cnt / n)
        print(f'    {cls_name:12s}: {cnt:5d}  ({100*cnt/n:.1f}%)  {bar}')
    maj = max(labels.count(i) for i in range(NUM_CLS))
    print(f'    Majority-class baseline: {maj/n:.4f}')
print()

# ── 2. Annotator agreement ────────────────────────────────────────────────────
print('── 2. Annotator Agreement ──')
unanimous, two_one, all_diff = 0, 0, 0
for pid in _full_train_ids:
    labels = [a['label'] for a in _data[pid]['annotators']]
    c = Counter(labels)
    if c.most_common(1)[0][1] == 3:   unanimous += 1
    elif c.most_common(1)[0][1] == 2: two_one   += 1
    else:                              all_diff  += 1
n_tr = len(_full_train_ids)
print(f'  Unanimous  (3/3): {unanimous:5d}  ({100*unanimous/n_tr:.1f}%)')
print(f'  Majority   (2/3): {two_one:5d}  ({100*two_one/n_tr:.1f}%)')
print(f'  No majority(1/1/1): {all_diff:4d}  ({100*all_diff/n_tr:.1f}%)')
print(f'  Fleiss kappa (reported in paper): ~0.45 (moderate)')
print()

# ── 3. Text length statistics ─────────────────────────────────────────────────
print('── 3. Text Length Statistics ──')
lengths_all = [len(_data[pid]['post_tokens']) for pid in _full_train_ids]
lengths_by_cls = {c: [] for c in LABEL_NAMES}
for pid in _full_train_ids:
    lbl = majority_label(_data[pid]['annotators'])
    lengths_by_cls[LABEL_NAMES[lbl]].append(len(_data[pid]['post_tokens']))
n_trunc = sum(l > MAX_LEN for l in lengths_all)
print(f'  Overall (train): mean={np.mean(lengths_all):.1f}  median={np.median(lengths_all):.0f}  '
      f'std={np.std(lengths_all):.1f}  max={max(lengths_all)}')
print(f'  Truncated (>{MAX_LEN} tokens): {n_trunc}/{len(lengths_all)}  ({100*n_trunc/len(lengths_all):.1f}%)')
for cls_name, lens in lengths_by_cls.items():
    print(f'  {cls_name}: mean={np.mean(lens):.1f}  median={np.median(lens):.0f}')
print()

# ── 4. Rationale coverage ─────────────────────────────────────────────────────
print('── 4. Rationale Coverage (train) ──')
rat_fracs_by_cls = {c: [] for c in LABEL_NAMES}
all_rat_fracs = []
for pid in _full_train_ids:
    ex  = _data[pid]
    lbl = majority_label(ex['annotators'])
    rat = avg_rationale(ex)
    frac_r = sum(rat) / len(rat) if rat else 0.0
    rat_fracs_by_cls[LABEL_NAMES[lbl]].append(frac_r)
    all_rat_fracs.append(frac_r)
print(f'  Overall mean rationale fraction: {np.mean(all_rat_fracs):.3f}')
print(f'  Posts with ≥1 rationale token: '
      f'{sum(f>0 for f in all_rat_fracs)}/{len(all_rat_fracs)} '
      f'({100*sum(f>0 for f in all_rat_fracs)/len(all_rat_fracs):.1f}%)')
for cls_name, fracs in rat_fracs_by_cls.items():
    print(f'  {cls_name}: mean={np.mean(fracs):.3f}')
print()

# ── 5. Top target communities ─────────────────────────────────────────────────
print('── 5. Top Target Communities (train, hate+offensive posts) ──')
target_counter = Counter()
for pid in _full_train_ids:
    ex  = _data[pid]
    lbl = majority_label(ex['annotators'])
    if lbl in [0, 2]:
        for a in ex['annotators']:
            for t in a.get('target', []):
                if t and t.lower() != 'none':
                    target_counter[t] += 1
for target, cnt in target_counter.most_common(10):
    print(f'  {target:<25s}: {cnt}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 0 — Dataset Characteristics (6-panel figure)
# ─────────────────────────────────────────────────────────────────────────────
cls_palette = ['#e15759', '#4e79a7', '#f28e2b']

fig = plt.figure(figsize=(18, 10))
fig.suptitle('HateXplain — Dataset Characteristics', fontsize=15, fontweight='bold')

# ── Panel 1: Class distribution — Train ───────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
tr_labels = _get_labels(_full_train_ids)
counts = [tr_labels.count(i) for i in range(NUM_CLS)]
bars = ax1.bar(LABEL_NAMES, counts, color=cls_palette, edgecolor='white', width=0.5)
for bar, c in zip(bars, counts):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
             str(c), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Class Distribution — Train', fontsize=11)
ax1.set_ylabel('Samples')
ax1.set_xticklabels(LABEL_NAMES, rotation=15)

# ── Panel 2: Class distribution — Test ────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
te_labels = _get_labels(_full_test_ids)
counts_t = [te_labels.count(i) for i in range(NUM_CLS)]
bars_t = ax2.bar(LABEL_NAMES, counts_t, color=cls_palette, edgecolor='white', width=0.5)
for bar, c in zip(bars_t, counts_t):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
             str(c), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_title('Class Distribution — Test', fontsize=11)
ax2.set_ylabel('Samples')
ax2.set_xticklabels(LABEL_NAMES, rotation=15)

# ── Panel 3: Token-length distribution ────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
ax3.hist(lengths_all, bins=40, color='#59a14f', edgecolor='white')
ax3.axvline(MAX_LEN, color='red', linestyle='--', linewidth=1.5, label=f'MAX_LEN={MAX_LEN}')
ax3.set_xlabel('Word count per post')
ax3.set_ylabel('Frequency')
ax3.set_title('Token-Length Distribution (train)', fontsize=11)
ax3.legend(fontsize=9)

# ── Panel 4: Annotator agreement pie ──────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
ax4.pie(
    [unanimous, two_one, all_diff],
    labels=['Unanimous\n(3/3)', 'Majority\n(2/3)', 'Split\n(1/1/1)'],
    autopct='%1.1f%%',
    colors=['#76b7b2', '#edc948', '#b07aa1'],
    startangle=90,
    textprops={'fontsize': 9},
)
ax4.set_title('Annotator Agreement (train)', fontsize=11)

# ── Panel 5: Rationale fraction boxplot by class ──────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
bp = ax5.boxplot([rat_fracs_by_cls[c] for c in LABEL_NAMES], patch_artist=True)
for patch, color in zip(bp['boxes'], cls_palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax5.set_xticklabels(LABEL_NAMES, rotation=15, fontsize=9)
ax5.set_ylabel('Fraction of rationale tokens')
ax5.set_title('Rationale Coverage by Class (train)', fontsize=11)
ax5.set_ylim(0, 1)

# ── Panel 6: Top target communities ───────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
top_targets = target_counter.most_common(8)
t_names  = [t[0][:18] for t in top_targets]
t_counts = [t[1]      for t in top_targets]
ax6.barh(t_names[::-1], t_counts[::-1], color='#ff9da7', edgecolor='white')
ax6.set_xlabel('Mention count (hate+offensive, train)')
ax6.set_title('Top Target Communities', fontsize=11)
ax6.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('fig_0_dataset_characteristics.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_0_dataset_characteristics.pdf')


## 2. Gate Modules — G1 through G5 + PNG

### Gate positions (applied inside `BertSelfAttention`)
| Name | Where | Formula |
|------|--------|---------|
| **G1** | Attention output (per head, before out projection) | `context *= G` |
| **G2** | Value vectors | `v *= G` |
| **G3** | Key vectors | `k *= G` |
| **G4** | Query vectors | `q *= G` |
| **G5** | Final context (after reshape, before BertSelfOutput) | `context *= G` |
| **PNG** | G1 + token-norm correction | `G = sigmoid(h@W - β·‖h‖·e)` |

In [ ]:
class GateParams(nn.Module):
    # Per-head gating parameters for one BERT self-attention layer.
    #   W    : (hidden_size, num_heads) -- maps hidden state to gate logit per head
    #   e    : (num_heads,)             -- per-head norm scale  [PNG only]
    #   beta : (1,)                     -- global norm weight    [PNG only]
    def __init__(self, dim, num_heads, png=False):
        super().__init__()
        self.png = png
        self.W   = nn.Parameter(torch.empty(dim, num_heads))
        nn.init.normal_(self.W, std=0.02)
        if png:
            self.e    = nn.Parameter(torch.ones(num_heads))
            self.beta = nn.Parameter(torch.zeros(1))

    def gate(self, x):
        # x : (B, N, dim) -> G : (B, N, num_heads)
        g = x @ self.W
        if self.png:
            x_norm = x.norm(dim=-1, keepdim=True) + 1e-6   # (B, N, 1)
            g = g - self.beta * x_norm * self.e             # broadcast over heads
        return torch.sigmoid(g)

In [ ]:
def _make_gate_forward(module, params, pos):
    # Returns a patched forward method for BERT BertSelfAttention.

    def patched_forward(self, hidden_states, attention_mask=None,
                        head_mask=None, encoder_hidden_states=None,
                        encoder_attention_mask=None, past_key_value=None,
                        output_attentions=False, **kwargs):
        bs, q_len, _ = hidden_states.size()
        H  = self.num_attention_heads
        dh = self.attention_head_size

        # ── Gate from input hidden states ──────────────────────────────────────
        G   = params.gate(hidden_states)         # (B, N, H)
        G_h = G.permute(0, 2, 1).unsqueeze(-1)  # (B, H, N, 1)

        # ── Q / K / V projections → (B, H, N, dh) ─────────────────────────────
        def _t(x, length):
            return x.view(bs, length, H, dh).permute(0, 2, 1, 3)

        q = _t(self.query(hidden_states), q_len)
        k = _t(self.key(hidden_states),   q_len)
        v = _t(self.value(hidden_states), q_len)

        if pos == 'G4': q = q * G_h
        if pos == 'G3': k = k * G_h
        if pos == 'G2': v = v * G_h

        # ── Scaled dot-product attention ────────────────────────────────────────
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(dh)
        if attention_mask is not None:
            scores = scores + attention_mask
        attn_probs = F.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        context = torch.matmul(attn_probs, v)  # (B, H, N, dh)

        if pos in ('G1', 'PNG'):
            context = context * G_h

        # ── Reshape to (B, N, all_head_size) ────────────────────────────────────
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(bs, q_len, self.all_head_size)

        if pos == 'G5':
            context = (
                context.view(bs, q_len, H, dh) * G.unsqueeze(-1)
            ).view(bs, q_len, self.all_head_size)

        # ── Capture flags for visualisation ─────────────────────────────────────
        if getattr(self, '_capture_gate', False):
            self._last_gate   = G.detach().cpu()
            self._last_x_norm = hidden_states.norm(dim=-1).detach().cpu()
        if getattr(self, '_capture_attn', False):
            self._last_attn = attn_probs.detach().cpu()

        return (context, attn_probs)

    return types.MethodType(patched_forward, module)

In [ ]:
def _is_bert_self_attn(mod):
    return all(hasattr(mod, a) for a in
               ['query', 'key', 'value', 'num_attention_heads', 'attention_head_size', 'dropout'])


def inject_gates(model, pos, png=False):
    # Monkey-patch every BertSelfAttention module with a gate.
    # Returns nn.ModuleList of GateParams (one per attention layer).
    dim = model.config.hidden_size
    H   = model.config.num_attention_heads
    gate_list = []

    for name, mod in model.named_modules():
        if _is_bert_self_attn(mod):
            p = GateParams(dim, H, png=png)
            p = p.to(next(mod.parameters()).device)
            mod.forward = _make_gate_forward(mod, p, pos)
            gate_list.append(p)
            print(f'  Injected {pos} gate into {name}')

    gate_params = nn.ModuleList(gate_list)
    model.gate_params = gate_params
    return gate_params

## 3. Model & Training Helpers

In [ ]:
def build_model(pos='baseline', png=False, num_classes=NUM_CLS):
    # Load BERT-base with FROZEN backbone.
    # Only classifier, bert.pooler, and gate_params are trained.
    m = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )

    if pos != 'baseline':
        print(f'Injecting {pos} gates (PNG={png}):')
        inject_gates(m, pos=pos, png=png)

    m = m.to(DEVICE)

    # ── Freeze all backbone parameters ────────────────────────────────────────
    for p in m.parameters():
        p.requires_grad = False

    # ── Unfreeze classifier head ───────────────────────────────────────────────
    for p in m.classifier.parameters():
        p.requires_grad = True

    # ── Unfreeze pooler ────────────────────────────────────────────────────────
    for p in m.bert.pooler.parameters():
        p.requires_grad = True

    # ── Unfreeze gate params (if any) ──────────────────────────────────────────
    if hasattr(m, 'gate_params'):
        for p in m.gate_params.parameters():
            p.requires_grad = True

    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    print(f'Trainable: {trainable:,} / {total:,}  (frozen backbone)')
    return m

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    criterion = nn.CrossEntropyLoss()

    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=attn)
        loss = criterion(out.logits, lbls)
        total_loss += loss.item()
        preds = out.logits.argmax(-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    return dict(loss=avg_loss, acc=acc, f1=f1, precision=prec, recall=rec)

In [ ]:
def train(model, epochs=EPOCHS, lr=LR_CLS, name='model', resume_ckpt=None):
    # Frozen backbone: only trainable params (classifier + pooler + gates) optimised.
    # resume_ckpt: path to an epoch-level checkpoint to resume from.
    opt_params = [p for p in model.parameters() if p.requires_grad]
    optimizer  = torch.optim.AdamW(opt_params, lr=lr, weight_decay=0.01)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)
    criterion  = nn.CrossEntropyLoss()

    log = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    start_epoch = 0

    # ── Resume from epoch-level checkpoint if available ────────────────────
    if resume_ckpt is not None and Path(resume_ckpt).exists():
        ckpt = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        log         = ckpt['log']
        start_epoch = ckpt['epoch']
        print(f'  Resumed from epoch {start_epoch} ({resume_ckpt})')

    for epoch in range(start_epoch, epochs):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_dl, desc=f'{name} E{epoch+1}/{epochs}', leave=False):
            ids  = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=attn)
            loss = criterion(out.logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(opt_params, 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_loss  = epoch_loss / len(train_dl)
        val_metrics = evaluate(model, val_dl)
        val_acc  = val_metrics['acc']
        val_f1   = val_metrics['f1']
        val_loss = val_metrics['loss']
        log['train_loss'].append(train_loss)
        log['val_loss'].append(val_loss)
        log['val_acc'].append(val_acc)
        log['val_f1'].append(val_f1)
        print(f'  Ep {epoch+1}: train_loss={train_loss:.4f}'
              f' | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}')

        # ── Epoch-level checkpoint ─────────────────────────────────────────
        sname_epoch = _safe_name(name)
        epoch_ckpt  = CKPT_DIR / f'{sname_epoch}_epoch{epoch+1}.pth'
        torch.save({
            'epoch':     epoch + 1,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'log':       log,
        }, epoch_ckpt)
        if epoch > start_epoch:
            prev = CKPT_DIR / f'{sname_epoch}_epoch{epoch}.pth'
            if prev.exists():
                prev.unlink()

    last_epoch_ckpt = CKPT_DIR / f'{_safe_name(name)}_epoch{epochs}.pth'
    if last_epoch_ckpt.exists():
        last_epoch_ckpt.unlink()

    return log

## 4. Experiment Configuration & Ablation Run

Edit `EXPERIMENTS` below to control exactly what runs.  
Each entry is a plain dict — comment out any row to skip it.  
The run loop **auto-skips** experiments whose `.pth` + `results.json` already exist,
so re-running the cell after a crash picks up where it left off.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment Configuration
# ─────────────────────────────────────────────────────────────────────────────
EXPERIMENTS = [
    dict(name='Baseline',          gate='baseline', epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G1 - Attn Output',  gate='G1',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G2 - Value Gate',   gate='G2',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G3 - Key Gate',     gate='G3',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G4 - Query Gate',   gate='G4',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='G5 - Final Output', gate='G5',       epochs=EPOCHS, is_png=False, lr=LR_CLS),
    dict(name='PNG (novel)',        gate='PNG',      epochs=EPOCHS, is_png=True,  lr=LR_CLS),
]

print(f'{len(EXPERIMENTS)} experiments configured:')
header = f'{"Name":<26} {"Gate":<10} {"Epochs":>6}  {"PNG":<5}  {"LR"}'
print(header)
print('-' * len(header))
for exp in EXPERIMENTS:
    print(f'{exp["name"]:<26} {exp["gate"]:<10} {exp["epochs"]:>6}  {str(exp["is_png"]):<5}  {exp["lr"]}')

In [ ]:
import json as _json
import traceback as _tb
import glob as _glob

RESULTS_FILE = CKPT_DIR / 'results.json'

# Resume: load previously saved results
if RESULTS_FILE.exists():
    with open(RESULTS_FILE) as _f:
        results = _json.load(_f)
    print(f'Resumed {len(results)} result(s) from {RESULTS_FILE}')
else:
    results = {}

train_logs = {}

for exp in EXPERIMENTS:
    vname  = exp['name']
    pos    = exp['gate']
    png    = exp['is_png']
    epochs = exp['epochs']
    lr     = exp['lr']
    sname  = _safe_name(vname)
    ckpt_file = CKPT_DIR / f'{sname}.pth'

    if vname in results and ckpt_file.exists():
        ta = results[vname]['acc']
        tf = results[vname]['f1']
        print(f'[SKIP] {vname}  (acc={ta:.4f}  f1={tf:.4f}) — checkpoint found')
        continue

    epoch_ckpts = sorted(_glob.glob(str(CKPT_DIR / f'{sname}_epoch*.pth')))
    resume_ckpt = epoch_ckpts[-1] if epoch_ckpts else None
    if resume_ckpt:
        print(f'[RESUME] {vname}  — found partial checkpoint: {resume_ckpt}')

    print(f'\n{"="*64}')
    print(f'  {vname}')
    print(f'  gate={pos}  png={png}  epochs={epochs}  lr={lr}')
    print(f'{"="*64}')

    try:
        model = build_model(pos=pos, png=png)
        log   = train(model, epochs=epochs, lr=lr, name=vname, resume_ckpt=resume_ckpt)

        test_metrics = evaluate(model, test_dl)
        results[vname]    = test_metrics
        train_logs[vname] = log

        torch.save(model.state_dict(), ckpt_file)

        with open(RESULTS_FILE, 'w') as _f:
            _json.dump(results, _f, indent=2)

        ta = test_metrics['acc']
        tf = test_metrics['f1']
        print(f'  TEST  acc={ta:.4f}  f1={tf:.4f}')
        print(f'  Saved checkpoint  -> {ckpt_file}')
        print(f'  Saved results.json -> {RESULTS_FILE}')

    except Exception as _e:
        print(f'  [ERROR] {vname} failed: {_e}')
        _tb.print_exc()
        print('  Skipping — continuing to next experiment...')

    finally:
        try:
            del model
        except NameError:
            pass
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

n_done = len(results)
n_total = len(EXPERIMENTS)
print(f'\nDone. {n_done}/{n_total} experiments completed.')

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
import json as _json
if not results and (CKPT_DIR / 'results.json').exists():
    with open(CKPT_DIR / 'results.json') as _f:
        results = _json.load(_f)
    print('Loaded results from results.json')

rows = []
for vname, m in results.items():
    rows.append({
        'Variant':    vname,
        'Acc':        round(m['acc'], 4),
        'F1 (macro)': round(m['f1'],  4),
        'Precision':  round(m['precision'], 4),
        'Recall':     round(m['recall'],    4),
    })

df_results = pd.DataFrame(rows).set_index('Variant')
display(df_results)

## 5. Figures

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig A — Bar chart: Accuracy & F1 across all 7 variants
# ─────────────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.1)

names  = list(results.keys())
accs   = [results[n]['acc'] for n in names]
f1s    = [results[n]['f1']  for n in names]
colors = ['#e15759' if 'PNG' in n else '#4e79a7' for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig A  |  Gate Position Ablation — Hate Speech Detection (Frozen Backbone)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, vals, title, ylbl in zip(
    axes,
    [accs, f1s],
    ['Test Accuracy', 'Test F1 (macro)'],
    ['Accuracy', 'F1'],
):
    bars = ax.bar(range(len(names)), vals, color=colors,
                  edgecolor='white', linewidth=0.6)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylbl, fontsize=12)
    ax.set_title(title, fontsize=12)
    lo = min(vals) - 0.03
    hi = max(vals) + 0.03
    ax.set_ylim(lo, hi)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4e79a7', label='Gate variants'),
    Patch(facecolor='#e15759', label='PNG (novel)'),
]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig_A_ablation_bar.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_A_ablation_bar.pdf')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig B — Training curves: Baseline vs G1 vs PNG  (val loss + val F1)
# ─────────────────────────────────────────────────────────────────────────────
highlight = ['Baseline', 'G1 - Attn Output', 'PNG (novel)']
palette   = {
    'Baseline':          '#4e79a7',
    'G1 - Attn Output':  '#59a14f',
    'PNG (novel)':       '#e15759',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig B  |  Training Curves — Baseline vs G1 vs PNG (Frozen)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, log_key, ylabel in zip(
    axes,
    ['val_loss', 'val_f1'],
    ['Val Loss', 'Val F1 (macro)'],
):
    for vname in highlight:
        if vname in train_logs:
            vals = train_logs[vname][log_key]
            ax.plot(
                range(1, len(vals) + 1), vals,
                label=vname, color=palette[vname],
                linewidth=2, marker='o', markersize=5,
            )
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(range(1, EPOCHS + 1))

plt.tight_layout()
plt.savefig('fig_B_training_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_B_training_curves.pdf')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig C — Layer x Head gate activation heatmap (PNG model)
# ─────────────────────────────────────────────────────────────────────────────
print('Rebuilding PNG model for gate visualisation...')
m_png = build_model(pos='PNG', png=True)

safe_png = _safe_name('PNG (novel)')
ckpt_path = CKPT_DIR / f'{safe_png}.pth'
if ckpt_path.exists():
    m_png.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))
    print('Loaded PNG checkpoint')
else:
    print('Checkpoint not found — using random gate params')

attn_modules = [mod for mod in m_png.modules() if _is_bert_self_attn(mod)]
for mod in attn_modules:
    mod._capture_gate = True

m_png.eval()
sample_batch = next(iter(val_dl))
with torch.no_grad():
    _ = m_png(
        input_ids=sample_batch['input_ids'].to(DEVICE),
        attention_mask=sample_batch['attention_mask'].to(DEVICE),
    )

n_layers = len(attn_modules)
n_heads  = attn_modules[0]._last_gate.shape[-1]
layer_head_gate = np.zeros((n_layers, n_heads))
for i, mod in enumerate(attn_modules):
    gate = mod._last_gate.numpy()
    layer_head_gate[i] = gate.mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    layer_head_gate,
    annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=[f'H{i+1}' for i in range(n_heads)],
    yticklabels=[f'L{i+1}' for i in range(n_layers)],
    vmin=0.0, vmax=1.0, ax=ax,
)
ax.set_xlabel('Attention Head', fontsize=12)
ax.set_ylabel('Transformer Layer', fontsize=12)
ax.set_title(
    'Fig C  |  PNG Gate Activations — Mean over Batch & Tokens (Layer x Head) [Frozen]',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.savefig('fig_C_gate_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_C_gate_heatmap.pdf')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig D — Scatter: gate suppression vs token norm
# ─────────────────────────────────────────────────────────────────────────────
all_norms, all_gates = [], []

for mod in attn_modules:
    gate  = mod._last_gate.numpy()
    xnorm = mod._last_x_norm.numpy()
    mean_gate = gate.mean(axis=-1)
    all_norms.append(xnorm.flatten())
    all_gates.append(mean_gate.flatten())

all_norms = np.concatenate(all_norms)
all_gates = np.concatenate(all_gates)

rng = np.random.default_rng(SEED)
idx = rng.choice(len(all_norms), size=min(3000, len(all_norms)), replace=False)
x_plot, y_plot = all_norms[idx], all_gates[idx]

r, p_val = pearsonr(x_plot, y_plot)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_plot, y_plot, alpha=0.25, s=8, c='#e15759', rasterized=True)
ax.set_xlabel('Token Hidden-State Norm  ||h||_2', fontsize=12)
ax.set_ylabel('PNG Gate Activation  G  (mean over heads)', fontsize=12)
ax.set_title(
    f'Fig D  |  PNG Gate Suppression vs Token Norm  (r={r:.3f}, p={p_val:.2e}) [Frozen]',
    fontsize=13, fontweight='bold',
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='G=0.5 (neutral)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_D_gate_vs_norm.pdf', bbox_inches='tight')
plt.show()
print(f'Saved fig_D_gate_vs_norm.pdf  (Pearson r={r:.3f})')
del m_png

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig E — Per-class F1 breakdown: Baseline vs G1 vs PNG  (3 classes)
# ─────────────────────────────────────────────────────────────────────────────
top_variants = [
    ('Baseline',          'baseline', False),
    ('G1 - Attn Output',  'G1',       False),
    ('PNG (novel)',        'PNG',      True),
]
per_class_f1 = {}

for vname, pos, png in top_variants:
    m = build_model(pos=pos, png=png)
    safe = _safe_name(vname)
    ckpt = CKPT_DIR / f'{safe}.pth'
    if ckpt.exists():
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
    m.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_dl:
            ids  = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            out  = m(input_ids=ids, attention_mask=attn)
            all_preds.extend(out.logits.argmax(-1).cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    per_class_f1[vname] = f1_score(all_labels, all_preds, average=None,
                                    labels=list(range(NUM_CLS)), zero_division=0)
    print(f'{vname}: {per_class_f1[vname]}')
    del m

n_classes = len(LABEL_NAMES)
x     = np.arange(n_classes)
width = 0.25
bar_colors = ['#4e79a7', '#59a14f', '#e15759']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (vname, _, _) in enumerate(top_variants):
    f1_cls = per_class_f1[vname]
    bars = ax.bar(
        x + (i - 1) * width, f1_cls,
        width, label=vname, color=bar_colors[i], edgecolor='white',
    )
    for bar, v in zip(bars, f1_cls):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES, fontsize=11)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_title('Fig E  |  Per-Class F1 — Baseline vs G1 vs PNG (Frozen)',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_E_per_class_f1.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_E_per_class_f1.pdf')

## Fig F — Token Attention Heatmap (BERT words) + Rationale Alignment
Shows which **words** each model attends to for individual examples, with a subplot row
showing the human rationale mask for direct visual comparison.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig F — Token-level Attention Heatmap with Human Rationale Comparison
# ─────────────────────────────────────────────────────────────────────────────

SPECIAL_TOKENS = {'[CLS]', '[SEP]', '[PAD]'}

def _capture_forward(mod, captured, layer_idx):
    import math as _math
    import types as _types
    import torch.nn.functional as _F

    def _fwd(self, hidden_states, attention_mask=None,
             head_mask=None, encoder_hidden_states=None,
             encoder_attention_mask=None, past_key_value=None,
             output_attentions=False, **kwargs):
        bs, q_len, _ = hidden_states.size()
        H  = self.num_attention_heads
        dh = self.attention_head_size

        def _t(x):
            return x.view(bs, q_len, H, dh).permute(0, 2, 1, 3)

        q = _t(self.query(hidden_states))
        k = _t(self.key(hidden_states))
        v = _t(self.value(hidden_states))

        scores = torch.matmul(q, k.transpose(-1, -2)) / _math.sqrt(dh)
        if attention_mask is not None:
            scores = scores + attention_mask
        attn_probs = _F.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        # Store CLS-to-all attention (mean over heads)
        captured[layer_idx] = attn_probs[0].mean(dim=0)[0].detach().cpu().numpy()

        context = torch.matmul(attn_probs, v)
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(bs, q_len, self.all_head_size)
        return (context, attn_probs)

    return _types.MethodType(_fwd, mod)


def get_cls_attention(model, input_ids, attention_mask):
    model.eval()
    attn_mods = [mod for mod in model.modules() if _is_bert_self_attn(mod)]

    if hasattr(model, 'gate_params'):
        for mod in attn_mods:
            mod._capture_attn = True
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)
        attn_layers = np.stack(
            [mod._last_attn[0].mean(dim=0)[0].cpu().numpy() for mod in attn_mods]
        )
        for mod in attn_mods:
            mod._capture_attn = False
    else:
        captured = {}
        saved_forwards = {i: mod.forward for i, mod in enumerate(attn_mods)}
        for i, mod in enumerate(attn_mods):
            mod.forward = _capture_forward(mod, captured, i)
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)
        for i, mod in enumerate(attn_mods):
            mod.forward = saved_forwards[i]
        attn_layers = np.stack([captured[i] for i in range(len(attn_mods))])

    return attn_layers, attn_layers.mean(axis=0)


def decode_tokens(input_ids_1d):
    toks = tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())
    return [t for t in toks if t != '[PAD]']


def strip_and_renorm(toks, attn):
    keep = [i for i, t in enumerate(toks) if t not in SPECIAL_TOKENS]
    toks_c = [toks[i] for i in keep]
    attn_c = attn[keep]
    total  = attn_c.sum()
    if total > 1e-9:
        attn_c = attn_c / total
    return toks_c, attn_c


# Select 4 examples: 1 per class (hatespeech x2, normal x1, offensive x1)
EXAMPLE_INDICES = []
targets = [0, 0, 1, 2]  # hatespeech, hatespeech, normal, offensive
for label_want in targets:
    for i in range(len(test_ds)):
        if test_ds[i]['label'].item() == label_want and i not in EXAMPLE_INDICES:
            EXAMPLE_INDICES.append(i)
            break

print('Selected test examples:')
for idx in EXAMPLE_INDICES:
    s = test_ds[idx]
    lbl = LABEL_NAMES[s['label'].item()]
    print(f'  [{idx}] label={lbl:12s}  tokens={s["post_tokens"][:10]}')


VARIANTS_F = [
    ('Baseline',         'baseline', False),
    ('G1 - Attn Output', 'G1',       False),
    ('PNG (novel)',       'PNG',      True),
]

models_f = {}
for vname, pos, png in VARIANTS_F:
    m = build_model(pos=pos, png=png)
    ckpt = CKPT_DIR / f'{_safe_name(vname)}.pth'
    if ckpt.exists():
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))
        print(f'Loaded checkpoint for {vname}')
    else:
        print(f'[WARN] No checkpoint for {vname} -- using random weights')
    m.eval()
    models_f[vname] = m


# Collect attention maps AND human rationale for each example
attn_maps = {v: {} for v, _, _ in VARIANTS_F}

for idx in EXAMPLE_INDICES:
    sample = test_ds[idx]
    ids    = sample['input_ids'].unsqueeze(0).to(DEVICE)
    mask   = sample['attention_mask'].unsqueeze(0).to(DEVICE)
    raw_toks = decode_tokens(sample['input_ids'])

    for vname, _, _ in VARIANTS_F:
        _, mean_attn = get_cls_attention(models_f[vname], ids, mask)
        toks_c, attn_c = strip_and_renorm(raw_toks, mean_attn[:len(raw_toks)])
        # Align with original post_tokens for rationale overlay
        post_toks = sample['post_tokens']
        rationale = np.array(sample['rationale'], dtype=float)
        # BPE tokens may differ from post_tokens — use post_tokens as labels for rationale
        # but use BPE-derived attention aligned to content tokens
        attn_maps[vname][idx] = {
            'bpe_toks':   toks_c,
            'bpe_attn':   attn_c,
            'post_toks':  post_toks,
            'rationale':  rationale,
        }


n_examples = len(EXAMPLE_INDICES)
n_variants = len(VARIANTS_F)
# Rows: n_examples variants rows + 1 rationale row per example
# Layout: (n_examples * 2) rows × n_variants cols
fig, axes = plt.subplots(
    n_examples * 2, n_variants,
    figsize=(6 * n_variants, 2.4 * n_examples * 2),
    constrained_layout=True,
)
fig.suptitle(
    'Fig F  |  BERT Token Attention Heatmap + Human Rationale [Frozen]\n'
    '(even rows: model attention; odd rows: human rationale mask)',
    fontsize=13, fontweight='bold',
)

lbl_colors = {'hatespeech': '#e15759', 'normal': '#4e79a7', 'offensive': '#f28e2b'}

for row_ex, idx in enumerate(EXAMPLE_INDICES):
    sample = test_ds[idx]
    label  = LABEL_NAMES[sample['label'].item()]
    color  = lbl_colors.get(label, 'black')

    for col, (vname, _, _) in enumerate(VARIANTS_F):
        data      = attn_maps[vname][idx]
        bpe_toks  = data['bpe_toks']
        bpe_attn  = data['bpe_attn']
        post_toks = data['post_toks']
        rationale = data['rationale']

        # --- Attention row ---
        ax_attn = axes[row_ex * 2, col]
        display_toks = bpe_toks[:30]  # cap to avoid crowding
        display_attn = bpe_attn[:30]
        sns.heatmap(
            display_attn[np.newaxis, :],
            ax=ax_attn,
            cmap='YlOrRd',
            xticklabels=display_toks,
            yticklabels=['attn'],
            cbar=(col == n_variants - 1),
            linewidths=0.3,
            linecolor='white',
        )
        ax_attn.set_xticklabels(display_toks, rotation=45, ha='right', fontsize=7)
        if row_ex == 0:
            ax_attn.set_title(vname, fontsize=11, fontweight='bold')
        if col == 0:
            ax_attn.set_ylabel(f'[{label}]\nattn', fontsize=8,
                               color=color, fontweight='bold')

        # --- Rationale row (use post_tokens as labels) ---
        ax_rat = axes[row_ex * 2 + 1, col]
        rat_display_toks = post_toks[:30]
        rat_display      = (rationale[:30] / (rationale[:30].max() + 1e-9))
        sns.heatmap(
            rat_display[np.newaxis, :],
            ax=ax_rat,
            cmap='Blues',
            xticklabels=rat_display_toks,
            yticklabels=['human'],
            cbar=(col == n_variants - 1),
            linewidths=0.3,
            linecolor='white',
            vmin=0, vmax=1,
        )
        ax_rat.set_xticklabels(rat_display_toks, rotation=45, ha='right', fontsize=7)
        if col == 0:
            ax_rat.set_ylabel('human\nrationale', fontsize=8)

plt.savefig('fig_F_token_attention_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_F_token_attention_heatmap.pdf')

for m in models_f.values():
    del m
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## Fig G — Rationale Alignment (HateXplain-specific)

HateXplain provides token-level **human rationale annotations** (words annotators deemed
responsible for the label). We measure how well each model's attention/gate activations
align with these human rationales using **Pearson r** between model scores and rationale masks.

- **Baseline / G1 / PNG**: compute alignment using mean CLS attention over content tokens
- **PNG gate alignment**: also compute alignment using PNG gate activations directly
- Report per-variant Pearson r with 95% CI bootstrapped from the test set

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig G — Rationale Alignment: model attention/gate vs human rationale masks
# ─────────────────────────────────────────────────────────────────────────────

def _make_attn_capture_forward(mod, store, layer_idx):
    # Monkey-patch a BertSelfAttention to capture CLS attention row.
    import math as _m, types as _t
    import torch.nn.functional as _F

    def _fwd(self, hidden_states, attention_mask=None,
             head_mask=None, encoder_hidden_states=None,
             encoder_attention_mask=None, past_key_value=None,
             output_attentions=False, **kwargs):
        bs, q_len, _ = hidden_states.size()
        H  = self.num_attention_heads
        dh = self.attention_head_size

        def _t2(x):
            return x.view(bs, q_len, H, dh).permute(0, 2, 1, 3)

        q = _t2(self.query(hidden_states))
        k = _t2(self.key(hidden_states))
        v = _t2(self.value(hidden_states))

        scores = torch.matmul(q, k.transpose(-1, -2)) / _m.sqrt(dh)
        if attention_mask is not None:
            scores = scores + attention_mask
        attn_probs = _F.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        # Store CLS row mean over heads: shape (seq_len,)
        store[layer_idx] = attn_probs[0].mean(dim=0)[0].detach().cpu().numpy()

        context = torch.matmul(attn_probs, v)
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(bs, q_len, self.all_head_size)
        return (context, attn_probs)

    return _t.MethodType(_fwd, mod)


def compute_rationale_alignment(model, dataset, n_samples=200, use_gate=False):
    # Pearson r between model token scores and binary human rationale masks.
    # use_gate=True  -> PNG gate activations (mean layers+heads)
    # use_gate=False -> CLS attention via monkey-patched capture forward
    model.eval()
    attn_mods = [mod for mod in model.modules() if _is_bert_self_attn(mod)]
    n_layers  = len(attn_mods)

    if use_gate:
        for mod in attn_mods:
            mod._capture_gate = True

    pearson_rs = []

    for idx in range(min(n_samples, len(dataset))):
        sample    = dataset[idx]
        post_toks = sample["post_tokens"]
        rationale = np.array(sample["rationale"], dtype=float)

        if rationale.sum() == 0 or len(post_toks) < 3:
            continue

        n_words = len(post_toks)
        ids  = sample["input_ids"].unsqueeze(0).to(DEVICE)
        mask = sample["attention_mask"].unsqueeze(0).to(DEVICE)

        if use_gate:
            with torch.no_grad():
                _ = model(input_ids=ids, attention_mask=mask)
            gate_layers = [mod._last_gate[0].mean(dim=-1).cpu().numpy()
                           for mod in attn_mods if hasattr(mod, "_last_gate")]
            if not gate_layers:
                continue
            mean_scores    = np.stack(gate_layers).mean(axis=0)
            content_scores = mean_scores[1:1 + n_words]
        else:
            # Monkey-patch every layer to capture CLS attention
            store = {}
            saved = {}
            for i, mod in enumerate(attn_mods):
                saved[i]   = mod.forward
                mod.forward = _make_attn_capture_forward(mod, store, i)
            with torch.no_grad():
                _ = model(input_ids=ids, attention_mask=mask)
            for i, mod in enumerate(attn_mods):
                mod.forward = saved[i]

            if not store:
                continue
            attn_layers    = np.stack([store[i] for i in range(n_layers) if i in store])
            mean_attn      = attn_layers.mean(axis=0)
            content_scores = mean_attn[1:1 + n_words]

        min_len = min(len(content_scores), len(rationale))
        if min_len < 3:
            continue
        s = content_scores[:min_len]
        r = rationale[:min_len]

        if r.std() < 1e-9 or s.std() < 1e-9:
            continue

        pr, _ = pearsonr(s, r)
        if not np.isnan(pr):
            pearson_rs.append(pr)

    return pearson_rs


VARIANTS_G = [
    ("Baseline",         "baseline", False, False),
    ("G1 - Attn Output", "G1",       False, False),
    ("PNG (attn)",        "PNG",      True,  False),
    ("PNG (gate)",        "PNG",      True,  True),
]

alignment_scores = {}
N_ALIGN_SAMPLES  = 300

for vname, pos, png, use_gate in VARIANTS_G:
    print(f"Computing rationale alignment for {vname}...")
    m = build_model(pos=pos, png=png)
    ckpt_key = "PNG (novel)" if "PNG" in vname else vname
    ckpt = CKPT_DIR / f"{_safe_name(ckpt_key)}.pth"
    if ckpt.exists():
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False))

    rs = compute_rationale_alignment(m, test_ds, n_samples=N_ALIGN_SAMPLES, use_gate=use_gate)
    alignment_scores[vname] = rs
    mean_r = np.mean(rs) if rs else float("nan")
    print(f"  {vname}: mean Pearson r = {mean_r:.4f}  (n={len(rs)} valid examples)")
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


# ── Plot Fig G ──────────────────────────────────────────────────────────────
variant_names = list(alignment_scores.keys())
mean_rs = [np.mean(v) if v else 0.0 for v in alignment_scores.values()]
sem_rs  = [np.std(v) / np.sqrt(max(len(v), 1)) for v in alignment_scores.values()]
bar_cols = ["#4e79a7", "#59a14f", "#e15759", "#b07aa1"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Fig G  |  Rationale Alignment — Model Token Scores vs Human Rationale Masks\n"
    "Metric: Pearson r between model scores and binary human rationale masks",
    fontsize=13, fontweight="bold",
)

ax = axes[0]
bars = ax.bar(range(len(variant_names)), mean_rs,
              color=bar_cols[:len(variant_names)], edgecolor="white",
              yerr=sem_rs, capsize=5)
ax.set_xticks(range(len(variant_names)))
ax.set_xticklabels(variant_names, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Mean Pearson r", fontsize=12)
ax.set_title("Rationale Alignment Score (mean ± SEM)", fontsize=11)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
for bar, v in zip(bars, mean_rs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax2 = axes[1]
data_for_plot   = [v for v in alignment_scores.values() if v]
labels_for_plot = [k for k, v in alignment_scores.items() if v]
if data_for_plot:
    parts = ax2.violinplot(data_for_plot, positions=range(len(data_for_plot)),
                           showmedians=True, showextrema=True)
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(bar_cols[i % len(bar_cols)])
        pc.set_alpha(0.7)
    ax2.set_xticks(range(len(labels_for_plot)))
    ax2.set_xticklabels(labels_for_plot, rotation=20, ha="right", fontsize=10)
    ax2.set_ylabel("Per-example Pearson r", fontsize=12)
    ax2.set_title("Distribution of Rationale Alignment (violin)", fontsize=11)
    ax2.axhline(0, color="gray", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.savefig("fig_G_rationale_alignment.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_G_rationale_alignment.pdf")
